#  Word2vec

__Автор задач: Блохин Н.В. (NVBlokhin@fa.ru)__

Материалы:
* Deep Learning with PyTorch (2020) Авторы: Eli Stevens, Luca Antiga, Thomas Viehmann
* https://radimrehurek.com/gensim/models/word2vec.html
* https://radimrehurek.com/gensim/auto_examples/tutorials/run_word2vec.html
* https://pytorch.org/text/stable/vocab.html
* https://github.com/OlgaChernytska/word2vec-pytorch
* https://www.baeldung.com/cs/nlps-word2vec-negative-sampling
* https://towardsdatascience.com/implementing-word2vec-in-pytorch-from-the-ground-up-c7fe5bf99889

## Задачи для совместного разбора

1\. Рассмотрите основные шаги подготовки данных для обучения skip-gram модели

In [2]:
import torch as th
import torch.nn as nn

In [1]:
text = "Рассмотрите основные шаги подготовки данных для обучения skip-gram модели"

In [3]:
tokens = text.lower().split()

In [4]:
x = []
y = []

for idx, token in enumerate(tokens):
  if idx == 0 or idx == len(tokens) - 1:
    continue

  x.append(token)
  y.append(tokens[idx - 1])

  x.append(token)
  y.append(tokens[idx + 1])

In [8]:
import pandas as pd

dataset = pd.DataFrame({"x": x, "y": y})
dataset["x"] = dataset["x"].map(tokens.index)
dataset["y"] = dataset["y"].map(tokens.index)

dataset.head(2)

,x,y
0,1,0
1,1,2


2\. Рассмотрите основные шаги по настройке skip-gram модели

In [9]:
x_t = th.tensor(dataset["x"])
y_t = th.tensor(dataset["y"])

In [10]:
vocab_size = len(tokens)
embeddings = nn.Embedding(
    num_embeddings=vocab_size,
    embedding_dim=16,
)
e = embeddings(x_t)
e.shape

torch.Size([14, 16])

In [12]:
fc = nn.Linear(
    in_features=embeddings.embedding_dim,
    out_features=vocab_size
)

out = fc(e)
out.shape

torch.Size([14, 9])

In [14]:
out[:3]

tensor([[ 0.8906, -1.1026,  0.0933,  0.0699,  1.0818, -0.0676,  0.3071,  0.6274,
         -0.9026],
        [ 0.8906, -1.1026,  0.0933,  0.0699,  1.0818, -0.0676,  0.3071,  0.6274,
         -0.9026],
        [-0.8069, -0.2578,  0.5081, -0.0309,  0.1166, -0.5963,  0.4329,  0.8493,
          0.3043]], grad_fn=<SliceBackward0>)

In [16]:
y_t[:3]

tensor([0, 2, 1])

In [17]:
criterion = nn.CrossEntropyLoss()

loss = criterion(out, y_t)
loss

tensor(2.1975, grad_fn=<NllLossBackward0>)

## Задачи для самостоятельного решения

In [48]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import nltk
from nltk.corpus import stopwords
import string

In [54]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
if torch.cuda.is_available():
    print(f"Используется GPU: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA не доступна, использование CPU.")

cuda
Используется GPU: NVIDIA GeForce RTX 3080 Ti


<p class="task" id="1"></p>

1\. Загрузите тексты новостей из файла `news_500.csv`. Удалите из текстов все знаки препинания и символы не из русского алфавита, приведите все слова к нижнему регистру и удалите стоп-слова.

- [ ] Проверено на семинаре

In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('russian'))

acceptable = 'абвгдеёжзийклмнопрстуфхцчшщъыьэюя '

df = pd.read_csv('news_500.csv')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nsedoff\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
def clean_text(text):
    text = text.lower()
    cleaned_text = ''

    for char in text:
        if char in acceptable:
            cleaned_text += char
        else:
            cleaned_text += ' '

    words = cleaned_text.split()
    filtered_words = [word for word in words if word not in stop_words]

    return ' '.join(filtered_words)

df['cleaned_text'] = df['text'].apply(clean_text)

In [9]:
print(df[['text', 'cleaned_text']].head())

                                                text  \
0  Президент России Владимир Путин считает, что к...   
1  Перед отъездом Я.Арафат провел совещание с пре...   
2  В связи с этим власти дали разрешение на прове...   
3  По словам Р.Вяхирева, это будет открытый тенде...   
4  Литературная премия, обладатель которой обычно...   

                                        cleaned_text  
0  президент россии владимир путин считает концеп...  
1  отъездом арафат провел совещание представителя...  
2  связи этим власти дали разрешение проведение с...  
3  словам р вяхирева это открытый тендер кого ден...  
4  литературная премия обладатель которой обычно ...  


<p class="task" id="2"></p>

2\. Настройте модель Word2Vec из пакета `gensim`. Для валидации выведите на экран информацию о ближайших словах для нескольких случайно выбранных токенов из обучающей выборки.

- [ ] Проверено на семинаре

In [10]:
tokenized_sentences = df['cleaned_text'].apply(lambda x: x.split()).tolist()

In [11]:
print(tokenized_sentences)

[['президент', 'россии', 'владимир', 'путин', 'считает', 'концепция', 'реформирования', 'армии', 'должна', 'готова', 'ноябрю', 'такое', 'заявление', 'сделал', 'совещанием', 'членов', 'совета', 'безопасности', 'рф', 'сообщили', 'рбк', 'администрации', 'президента', 'путин', 'также', 'отметил', 'эта', 'реформа', 'должна', 'проводиться', 'учетом', 'проблем', 'существующих', 'настоящее', 'время', 'вооруженных', 'силах', 'рф', 'одну', 'таких', 'проблем', 'путин', 'охарактеризовал', 'параллелизм', 'армейских', 'структур', 'основная', 'цель', 'реформирования', 'армии', 'сделать', 'эффективной', 'особенное', 'внимание', 'следует', 'уделить', 'социальным', 'вопросам', 'обеспечения', 'военнослужащих', 'президент', 'россии', 'заявил', 'реформирование', 'проводиться', 'учетом', 'основных', 'положений', 'предложенных', 'советом', 'безопасности', 'россии', 'вместе', 'путин', 'считает', 'массовых', 'сокращений', 'армии', 'хотя', 'отмечает', 'военные', 'нужды', 'россия', 'тратит', 'слишком', 'бюджетны

In [12]:
from gensim.models import Word2Vec
model = Word2Vec(tokenized_sentences, vector_size=100, window=5, min_count=1, workers=4)

In [13]:
vector = model.wv['президент']
print(vector)

[-1.79159269e-02  6.64596558e-02  2.33036559e-02  1.75238457e-02
 -6.58169901e-03 -8.47059786e-02  1.91922411e-02  9.55503508e-02
 -3.74445096e-02 -1.96223930e-02 -3.27501334e-02 -6.21632710e-02
 -1.49399377e-02  2.60712299e-02  1.96792223e-02 -3.27032618e-02
  3.90853547e-03 -4.73647974e-02 -1.20469155e-02 -1.03905767e-01
  3.08275931e-02  2.36262754e-02  3.90167199e-02 -3.49217094e-02
 -1.07336054e-02  1.94455497e-02 -5.28930612e-02 -1.37273595e-02
 -5.27183749e-02  3.00129247e-03  5.51930778e-02 -1.52457943e-02
  2.44921111e-02 -1.95485372e-02 -2.20287107e-02  1.76635869e-02
  1.66392997e-02 -2.25885045e-02 -4.24375534e-02 -7.91516080e-02
  1.57361105e-02 -2.94678695e-02 -4.28604372e-02  7.48252124e-03
  4.19575088e-02 -2.21337397e-02 -3.49018015e-02  1.52890384e-02
  1.73061225e-03  5.25659509e-02  2.10157782e-02 -5.57785146e-02
 -4.02308628e-02  8.50559492e-03 -3.21607366e-02  1.90393534e-02
  2.58217845e-02 -5.87271415e-06 -5.42496406e-02 -1.93096157e-02
  1.39755690e-02  1.73269

In [14]:
similar_words = model.wv.most_similar('президент', topn=3)
print(similar_words)

[('сегодня', 0.9833205938339233), ('рф', 0.9827442765235901), ('отметил', 0.9821828603744507)]


In [16]:
import random
words = list(model.wv.index_to_key)

In [27]:
random_words = random.sample(words, 5)
for word in random_words:
    print(f"Ближайшие слова для '{word}':")
    similar_words = model.wv.most_similar(word, topn=5)
    for similar_word, similarity in similar_words:
        print(f"  {similar_word} (схожесть: {similarity:.4f})")
    print()

Ближайшие слова для 'высокий':
  мотивы (схожесть: 0.4290)
  местным (схожесть: 0.3715)
  испытывающая (схожесть: 0.3683)
  карло (схожесть: 0.3619)
  остановлены (схожесть: 0.3534)

Ближайшие слова для 'краевых':
  обогатительного (схожесть: 0.3729)
  бизнесом (схожесть: 0.3711)
  выполняется (схожесть: 0.3649)
  убито (схожесть: 0.3537)
  аварии (схожесть: 0.3477)

Ближайшие слова для 'проинформировала':
  экспортной (схожесть: 0.3675)
  вынудивших (схожесть: 0.3632)
  коммерции (схожесть: 0.3513)
  заменяются (схожесть: 0.3458)
  уверенностью (схожесть: 0.3340)

Ближайшие слова для 'идее':
  гражданину (схожесть: 0.3719)
  указаны (схожесть: 0.3638)
  равнинную (схожесть: 0.3413)
  изображен (схожесть: 0.3393)
  иммиграции (схожесть: 0.3345)

Ближайшие слова для 'масхадова':
  побывать (схожесть: 0.3906)
  полтора (схожесть: 0.3795)
  вдв (схожесть: 0.3763)
  отправится (схожесть: 0.3678)
  работа (схожесть: 0.3612)



<p class="task" id="3"></p>

3. Опишите класс `W2VDataset`, который реализует в себе логику получения контекстного окна для обучения skip-gram модели. При создании словаря игнорируйте токены, которые встретились меньше 20 раз. Продемонстрируйте пример работы.

![image.png](https://github.com/OlgaChernytska/word2vec-pytorch/raw/main/docs/skipgram_overview.png)

- [ ] Проверено на семинаре

In [36]:
from collections import defaultdict

class W2VDataset:
    def __init__(self, sentences, window_size, min_count=20):
        """
        Инициализация Dataset для обучения Word2Vec (Skip-Gram)
        
        :param sentences: Список предложений (каждое предложение - список слов)
        :param window_size: Размер контекстного окна
        :param min_count: Минимальное количество встречаемости слова, чтобы оно попало в словарь
        """
        self.sentences = sentences
        self.window_size = window_size
        self.min_count = min_count
        self.vocab = self.build_vocab(sentences)
        self.word_pairs = self.create_word_pairs()

    def build_vocab(self, sentences):
        """
        Строим словарь, который содержит частоту каждого слова в корпусе,
        игнорируя слова, которые встречаются меньше min_count раз.
        
        :param sentences: Список предложений
        :return: Словарь, где ключи - слова, а значения - их частоты
        """
        vocab = defaultdict(int)
        for sentence in sentences:
            for word in sentence:
                vocab[word] += 1
        
        # Убираем слова, которые встречаются меньше min_count раз
        vocab = {word: count for word, count in vocab.items() if count >= self.min_count}
        
        return vocab

    def create_word_pairs(self):
        """
        Создаем все возможные пары (target, context) для обучения модели,
        игнорируя слова, которых нет в словаре (в которых частота меньше min_count).
        
        :return: Список пар (target, context)
        """
        word_pairs = []
        
        for sentence in self.sentences:
            sentence_len = len(sentence)
            for i, target_word in enumerate(sentence):
                if target_word not in self.vocab:
                    continue  # Пропускаем слова, которых нет в словаре
                
                # Для каждого слова, получаем его контекстные слова в пределах окна
                left = max(0, i - self.window_size)
                right = min(sentence_len, i + self.window_size + 1)
                
                # Перебираем все слова в контексте, кроме самого целевого
                for j in range(left, right):
                    if i != j:  # Не берем саму цель
                        context_word = sentence[j]
                        if context_word in self.vocab:
                            word_pairs.append((target_word, context_word))
        
        return word_pairs

    def get_batch(self, batch_size):
        """
        Получаем случайную партию (batch) из обучающих данных.
        
        :param batch_size: Размер батча
        :return: Список случайных пар (target, context)
        """
        batch = random.sample(self.word_pairs, batch_size)
        return batch


In [46]:
sentences = tokenized_sentences

dataset = W2VDataset(sentences, window_size=2, min_count=1)

batch = dataset.get_batch(4)
print("Случайный батч:", batch)

print("\nСловарь:")
print(dataset.vocab)

Случайный батч: [('военнослужащим', 'жилья'), ('украинцам', 'открыто'), ('финансовых', 'обвиняло'), ('идти', 'речь')]

Словарь:
{'президент': 93, 'россии': 453, 'владимир': 90, 'путин': 54, 'считает': 115, 'концепция': 8, 'реформирования': 9, 'армии': 15, 'должна': 47, 'готова': 10, 'ноябрю': 1, 'такое': 44, 'заявление': 29, 'сделал': 9, 'совещанием': 1, 'членов': 28, 'совета': 94, 'безопасности': 76, 'рф': 358, 'сообщили': 147, 'рбк': 530, 'администрации': 62, 'президента': 117, 'также': 323, 'отметил': 220, 'эта': 24, 'реформа': 10, 'проводиться': 7, 'учетом': 18, 'проблем': 16, 'существующих': 2, 'настоящее': 56, 'время': 231, 'вооруженных': 16, 'силах': 2, 'одну': 27, 'таких': 15, 'охарактеризовал': 3, 'параллелизм': 1, 'армейских': 1, 'структур': 12, 'основная': 7, 'цель': 12, 'сделать': 11, 'эффективной': 1, 'особенное': 1, 'внимание': 20, 'следует': 25, 'уделить': 1, 'социальным': 1, 'вопросам': 17, 'обеспечения': 9, 'военнослужащих': 15, 'заявил': 207, 'реформирование': 2, 'осн

<p class="task" id="4"></p>

4\. Реализуйте и настройте skip-gram модель. Перед началом обучения выберите случайным образом несколько слов из датасета и для каждого из них выведите на экран 3 ближайших слова в смысле косинусной близости между эмбеддингами. В процессе настройки для валидации периодически выводите на экран информацию о ближайших словах для этих слов. Выведите на экран график значения функции потерь в зависимости от номера эпохи.  

![image.png](https://github.com/OlgaChernytska/word2vec-pytorch/raw/main/docs/skipgram_detailed.png)

- [ ] Проверено на семинаре

In [74]:

# Гиперпараметры
embedding_dim = 100  # Размер эмбеддингов
batch_size = 32       # Размер батча
window_size = 16      # Размер контекстного окна

# Словарь и индексы
word_to_idx = {word: idx for idx, word in enumerate(dataset.vocab.keys())}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

In [75]:
# Модель Skip-Gram
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramModel, self).__init__()
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)  # Входной слой
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)  # Выходной слой

    def forward(self, target, context):
        target_embeds = self.in_embed(target)
        context_embeds = self.out_embed(context)
        return torch.matmul(target_embeds, context_embeds.t())  # Предсказания


In [76]:
# Инициализация модели
vocab_size = len(word_to_idx)
model = SkipGramModel(vocab_size, embedding_dim).to(device)

# Оптимизатор и функция потерь
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

In [77]:
# Функция для обучения модели с дополнительной проверкой
def train(model, dataset, batch_size, epochs):
    model.train()
    for epoch in range(epochs):
        batch = dataset.get_batch(batch_size)
        target_words, context_words = zip(*batch)

        # Преобразуем слова в индексы
        target_indices = [word_to_idx[word] for word in target_words]
        context_indices = [word_to_idx[word] for word in context_words]

        # Проверка максимальных индексов
        print(f"Максимальный индекс целевого слова: {max(target_indices)}")
        print(f"Максимальный индекс контекстного слова: {max(context_indices)}")

        # Проверка индексов
        assert max(target_indices) < vocab_size, f"Target index out of bounds: {max(target_indices)}"
        assert max(context_indices) < vocab_size, f"Context index out of bounds: {max(context_indices)}"

        # Переводим индексы в тензоры и отправляем на устройство
        target_words_idx = torch.tensor(target_indices).to(device)
        context_words_idx = torch.tensor(context_indices).to(device)

        optimizer.zero_grad()

        predictions = model(target_words_idx, context_words_idx)

        # Проверка размера предсказаний
        print(f"Размер предсказаний: {predictions.size()}")  # (batch_size, vocab_size)

        # Вычисление потерь
        loss = criterion(predictions, context_words_idx)

        # Обратное распространение и оптимизация
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f"Эпоха {epoch + 1}, Потери: {loss.item()}")

In [81]:
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import defaultdict

# Гиперпараметры
embedding_dim = 100  # Размер эмбеддингов
batch_size = 32      # Размер батча
window_size = 16     # Размер контекстного окна

# Словарь и индексы
word_to_idx = {word: idx for idx, word in enumerate(dataset.vocab.keys())}
idx_to_word = {idx: word for word, idx in word_to_idx.items()}

# Модель Skip-Gram
class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super(SkipGramModel, self).__init__()
        self.in_embed = nn.Embedding(vocab_size, embedding_dim)  # Входной слой
        self.out_embed = nn.Embedding(vocab_size, embedding_dim)  # Выходной слой

    def forward(self, target, context):
        target_embeds = self.in_embed(target)
        context_embeds = self.out_embed(context)
        return torch.matmul(target_embeds, context_embeds.t())  # Предсказания

# Инициализация модели
vocab_size = len(word_to_idx)
model = SkipGramModel(vocab_size, embedding_dim).to(device)

# Оптимизатор и функция потерь
optimizer = optim.SGD(model.parameters(), lr=0.1)
criterion = nn.CrossEntropyLoss()

# Функция для обучения модели с дополнительной проверкой
def train(model, dataset, batch_size, epochs):
    model.train()
    for epoch in range(epochs):
        batch = dataset.get_batch(batch_size)
        target_words, context_words = zip(*batch)

        # Преобразуем слова в индексы
        target_indices = [word_to_idx[word] for word in target_words]
        context_indices = [word_to_idx[word] for word in context_words]

        # Печать максимальных индексов для диагностики
        print(f"Максимальный индекс целевого слова: {max(target_indices)}")
        print(f"Максимальный индекс контекстного слова: {max(context_indices)}")

        # Печать некоторых индексов для диагностики
        print(f"Некоторые индексы целевых слов: {target_indices[:5]}")
        print(f"Некоторые индексы контекстных слов: {context_indices[:5]}")

        # Проверка, что индексы не выходят за пределы
        if max(target_indices) >= vocab_size:
            print(f"Ошибка: целевой индекс {max(target_indices)} выходит за пределы!")
        if max(context_indices) >= vocab_size:
            print(f"Ошибка: контекстный индекс {max(context_indices)} выходит за пределы!")

        # Проверка индексов на выход за пределы
        assert max(target_indices) < vocab_size, f"Target index out of bounds: {max(target_indices)}"
        assert max(context_indices) < vocab_size, f"Context index out of bounds: {max(context_indices)}"

        # Переводим индексы в тензоры и отправляем на устройство
        target_words_idx = torch.tensor(target_indices).to(device)
        context_words_idx = torch.tensor(context_indices).to(device)

        optimizer.zero_grad()

        predictions = model(target_words_idx, context_words_idx)

        # Проверка размера предсказаний
        print(f"Размер предсказаний: {predictions.size()}")  # (batch_size, vocab_size)

        # Вычисление потерь
        loss = criterion(predictions, context_words_idx)

        # Обратное распространение и оптимизация
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f"Эпоха {epoch + 1}, Потери: {loss.item()}")

# Обучаем модель
epochs = 1000  # Количество эпох
train(model, dataset, batch_size, epochs)


Максимальный индекс целевого слова: 15575
Максимальный индекс контекстного слова: 14588
Некоторые индексы целевых слов: [4229, 13559, 645, 4297, 1032]
Некоторые индексы контекстных слов: [14588, 1698, 4231, 4296, 3436]
Размер предсказаний: torch.Size([32, 32])


IndexError: Target 14588 is out of bounds.

<p class="task" id="5"></p>

5\. Посчитайте частоту совместного использования слов (с использованием контекстных окон ширины 5). Выберите одно слово ("центральное слово") случайным образом. Найдите топ-10 слов, наиболее часто используемых вместе с ним. Для каждой такой пары найдите ранг совместно используемого слова (в порядке увеличения евликлидова расстояния между данным и центральным словом). Выясните, как скоррелированы ранги по расстоянию между эмбеддингами и по частоте совместного использования.

- [ ] Проверено на семинаре